In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [2]:
os.makedirs("../models", exist_ok=True)
print("Models folder created successfully.")

Models folder created successfully.


In [3]:
DATA_PATH = "../data/processed/final_processed_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (52930, 24)


,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating,ContinentId,RegionId,CountryId,...,AttractionAddress,AttractionType,CityId_city,CityName,CountryId_city,Country,RegionId_country,Region,ContinentId_region,Continent
0,3,70456,2022,10,2,640,5,5,21,163,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,United Kingdom,21,Western Europe,5,Europe
1,8,7567,2022,10,4,640,5,2,8,48,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Canada,8,Northern America,2,America
2,9,79069,2022,10,3,640,5,2,9,54,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Brazil,9,South America,2,America
3,10,31019,2022,10,3,640,3,5,17,135,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Switzerland,17,Central Europe,5,Europe
4,15,43611,2022,10,2,640,3,5,21,163,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,United Kingdom,21,Western Europe,5,Europe


In [4]:
print("Available Columns:")
print(df.columns.tolist())

Available Columns:
['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitMode', 'AttractionId', 'Rating', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'CityId_city', 'CityName', 'CountryId_city', 'Country', 'RegionId_country', 'Region', 'ContinentId_region', 'Continent']


In [5]:
required_columns = ["UserId", "AttractionId", "Rating"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

rec_df = df[required_columns].copy()

print("Recommendation dataset shape:", rec_df.shape)
rec_df.head()

Recommendation dataset shape: (52930, 3)


,UserId,AttractionId,Rating
0,70456,640,5
1,7567,640,5
2,79069,640,5
3,31019,640,3
4,43611,640,3


In [6]:
rec_df = rec_df.dropna()

print("Shape after removing missing values:", rec_df.shape)

Shape after removing missing values: (52930, 3)


In [7]:
user_item_matrix = rec_df.pivot_table(
    index="UserId",
    columns="AttractionId",
    values="Rating",
    fill_value=0
)

print("User-Item Matrix Shape:", user_item_matrix.shape)
user_item_matrix.head()

User-Item Matrix Shape: (33530, 30)


AttractionId,369,481,640,650,673,737,748,749,824,841,...,1133,1137,1166,1171,1220,1225,1238,1278,1280,1297
UserId,,,,,,,,,,,,,,,,,,,,,
14,0.0,0.0,4.00,0.0,0.0,0.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,0.0,5.0,4.25,0.0,0.0,0.0,5.0,0.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,0.0,0.0,0.00,0.0,0.0,0.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
item_similarity = cosine_similarity(
    user_item_matrix.T
)

print("Item similarity matrix created successfully.")
print("Shape:", item_similarity.shape)

Item similarity matrix created successfully.
Shape: (30, 30)


In [9]:
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

item_similarity_df.iloc[:5, :5]

AttractionId,369,481,640,650,673
AttractionId,,,,,
369,1.000000,0.051794,0.071406,0.057954,0.067234
481,0.051794,1.000000,0.068480,0.043912,0.055698
640,0.071406,0.068480,1.000000,0.089273,0.089883
650,0.057954,0.043912,0.089273,1.000000,0.061950
673,0.067234,0.055698,0.089883,0.061950,1.000000


In [10]:
def recommend_attractions(attraction_id, top_n=5):
    """
    Recommend similar attractions based on attraction ID.
    """

    if attraction_id not in item_similarity_df.index:
        print("Attraction ID not found.")
        return pd.DataFrame()

    similar_scores = item_similarity_df[attraction_id].sort_values(
        ascending=False
    )

    # Remove the same attraction
    similar_scores = similar_scores.drop(attraction_id)

    recommendations = similar_scores.head(top_n)

    return pd.DataFrame({
        "AttractionId": recommendations.index,
        "Similarity Score": recommendations.values
    })

In [11]:
sample_attraction_id = rec_df["AttractionId"].iloc[0]

print("Sample Attraction ID:", sample_attraction_id)
print("\nTop 5 Recommended Attractions:")

recommendations = recommend_attractions(
    attraction_id=sample_attraction_id,
    top_n=5
)

recommendations

Sample Attraction ID: 640

Top 5 Recommended Attractions:


,AttractionId,Similarity Score
0,748,0.274019
1,749,0.174610
2,737,0.116497
3,841,0.110575
4,824,0.104087


In [12]:
joblib.dump(
    item_similarity_df,
    "../models/item_similarity_df.pkl"
)

joblib.dump(
    user_item_matrix,
    "../models/user_item_matrix.pkl"
)

recommender_data = {
    "item_similarity_df": item_similarity_df,
    "user_item_matrix": user_item_matrix
}

joblib.dump(
    recommender_data,
    "../models/recommender.pkl"
)

print("Recommendation artifacts saved successfully.")
print("Files created:")
print("1. ../models/recommender.pkl")
print("2. ../models/item_similarity_df.pkl")
print("3. ../models/user_item_matrix.pkl")

Recommendation artifacts saved successfully.
Files created:
1. ../models/recommender.pkl
2. ../models/item_similarity_df.pkl
3. ../models/user_item_matrix.pkl


In [13]:
print("=" * 60)
print("Recommendation System Completed Successfully")
print("=" * 60)

print("Artifacts Saved:")
print("1. ../models/recommender.pkl")
print("2. ../models/item_similarity_df.pkl")
print("3. ../models/user_item_matrix.pkl")

print("\nThe recommendation system is ready for use in the Streamlit application.")

Recommendation System Completed Successfully
Artifacts Saved:
1. ../models/recommender.pkl
2. ../models/item_similarity_df.pkl
3. ../models/user_item_matrix.pkl

The recommendation system is ready for use in the Streamlit application.
